In [ ]:
from pathlib import Path
ROOT = Path('..').resolve()
DATA_DIR = ROOT / 'data'
IMG_DIR = DATA_DIR / 'images'

In [ ]:
import pandas as pd
import torch

data = pd.read_csv(f'{DATA_DIR}/dataset_final.csv')
print(data.head())

In [ ]:
import matplotlib.pyplot as plt
emotion_distribution = data['emotion'].value_counts()
plt.figure(figsize=(8, 5))
emotion_distribution.plot(kind='bar')
plt.title('Emotion Distribution')
plt.xlabel('Emotion')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
print(emotion_distribution.describe())


utterance_lengths = data['utterance'].apply(lambda x: len(str(x).split()))
plt.figure(figsize=(8, 5))
plt.hist(utterance_lengths, bins=20)
plt.title('Utterance Length Distribution')
plt.xlabel('Number of Words')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()
print(utterance_lengths.describe())

image_coverage = data['filename'].nunique()
print(f'Unique images: {image_coverage}')
actual_images_count = len(list(IMG_DIR.glob('*.jpg')))
print(f'Actual images in directory: {actual_images_count}')

invalid_images = set(data['filename']) - set(img.name for img in IMG_DIR.glob('*.jpg'))
print(f'Invalid image references: {len(invalid_images)}')

annotations_per_image = data.groupby('filename').size()
print(annotations_per_image.describe())

print(data.shape)

In [ ]:
from collections import Counter
import string
MIN_FREQUENCY = 2

tokenized_utterances = data['utterance'].apply(lambda x: [word.strip(string.punctuation) for word in str(x).lower().split()])
word_counts = Counter(word for utterance in tokenized_utterances for word in utterance)
vocab = {word for word in word_counts if word_counts[word] >= MIN_FREQUENCY}
vocab.update({'<PAD>', '<UNK>', '<SOS>', '<EOS>'})

word2idx = {word: idx for idx, word in enumerate(sorted(vocab))}
idx2word = {idx: word for word, idx in word2idx.items()}

print(f'Vocabulary size: {len(vocab)}')
print(f'Index for special tokens: <PAD>={word2idx["<PAD>"]}, <UNK>={word2idx["<UNK>"]}, <SOS>={word2idx["<SOS>"]}, <EOS>={word2idx["<EOS>"]}')

sample_utterance = data['utterance'].iloc[0].lower()
tokenized = [word.strip(string.punctuation) for word in sample_utterance.split()]
indices = [word2idx.get(word, word2idx['<UNK>']) for word in tokenized]
print(f'Original: {sample_utterance}')
print(f'Tokenized: {tokenized}')
print(f'Indices: {indices}')
reconstructed = ' '.join(idx2word[idx] for idx in indices)
print(f'Reconstructed: {reconstructed}')


In [ ]:
from torchvision import transforms

image_transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225])
    ])

In [ ]:
from PIL import Image

class ArtemisDataset(torch.utils.data.Dataset):
    def __init__(self, data, word2idx, img_dir, image_transform, max_seq_length=40):
        self.data = data
        self.word2idx = word2idx
        self.img_dir = img_dir
        self.image_transform = image_transform
        self.max_seq_length = max_seq_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, key):
        row = self.data.iloc[key]
        img_path = f'{self.img_dir}/{row["filename"]}'
        img = Image.open(img_path).convert('RGB')
        img_tensor = self.image_transform(img)
        
        utterance = [word.strip(string.punctuation) for word in str(row['utterance']).lower().split()]
        indices = [self.word2idx.get(word, self.word2idx['<UNK>']) for word in utterance]
        indices = [self.word2idx['<SOS>']] + indices + [self.word2idx['<EOS>']]
        if len(indices) < self.max_seq_length:
            indices += [self.word2idx['<PAD>']] * (self.max_seq_length - len(indices))
        else:
            indices = indices[:self.max_seq_length-1] + [self.word2idx['<EOS>']]
        
        return img_tensor, torch.tensor(indices)
    

In [ ]:
dataset = ArtemisDataset(data, word2idx, IMG_DIR, image_transform)
print(f'Dataset size: {len(dataset)}')
sample_img, sample_indices = dataset[0]
print(f'Sample image tensor shape: {sample_img.shape}')
print(f'Sample indices: {sample_indices}')
print(f'Sample reconstructed: {" ".join(idx2word[idx.item()] for idx in sample_indices if idx.item() in idx2word)}')

In [ ]:
from sklearn.model_selection import train_test_split
train_data, test_data = train_test_split(data, test_size=0.1, random_state=42)
train_data, val_data = train_test_split(train_data, test_size=0.111, random_state=42)

train_dataset = ArtemisDataset(train_data, word2idx, IMG_DIR, image_transform)
val_dataset = ArtemisDataset(val_data, word2idx, IMG_DIR, image_transform)
test_dataset = ArtemisDataset(test_data, word2idx, IMG_DIR, image_transform)

In [ ]:
BATCH_SIZE = 32
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE)

In [ ]:
test_batch = next(iter(train_loader))
test_images, test_sequences = test_batch
print(test_images.shape)
print(test_sequences.shape)